# 24 — OpenRouter Structured Output

**Pattern:** Provider-agnostic structured output — OpenRouter unified API + `openai` SDK.  
**Key insight:** Swap the model string; every other line of code stays the same.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/24-openrouter-structured-output/openrouter_structured_output_workbook.ipynb)

In [ ]:
%pip install -q openai pydantic python-dotenv

In [ ]:
import os
from google.colab import userdata

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

## 1. The Pattern

LangChain's `with_structured_output()` is convenient, but it hides the mechanism.  
The underlying primitive is `client.beta.chat.completions.parse()` from the OpenAI SDK.

OpenRouter exposes the same interface at `https://openrouter.ai/api/v1`, so you can route to **any supported model** without changing your code.

```
OpenAI(base_url="https://openrouter.ai/api/v1", api_key=KEY)
  → client.beta.chat.completions.parse(model="openai/gpt-4o-mini", response_format=MySchema)
  → completion.choices[0].message.parsed  # MySchema instance
```

Change `model=` to `"anthropic/claude-3-haiku"` — nothing else changes.

## 2. Schema

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field


class EmailTriage(BaseModel):
    urgency: Literal["high", "medium", "low"] = Field(
        description="How urgently this email needs a response."
    )
    category: Literal["billing", "technical", "general", "spam"] = Field(
        description="The primary topic category of the email."
    )
    summary: str = Field(
        description="One-sentence plain-English summary of what the email is about."
    )
    recommended_action: str = Field(
        description="Concrete next step the recipient should take."
    )

## 3. Workflow

In [ ]:
from openai import OpenAI


def classify(email_text: str, model: str = "openai/gpt-4o-mini") -> EmailTriage:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are an email triage assistant. Classify the email and recommend an action.",
            },
            {"role": "user", "content": email_text},
        ],
        response_format=EmailTriage,
    )
    return completion.choices[0].message.parsed

## 4. Run — Two Sample Emails

In [ ]:
samples = [
    (
        "Overdue invoice",
        """Subject: URGENT - Invoice #4821 overdue 30 days, account suspended
From: accounts@acme-corp.com

Our account has been suspended due to unpaid invoice #4821 ($4,200).
Payment was due May 1st -- now 30 days overdue.""",
    ),
    (
        "Feature request",
        """Subject: Idea - dark mode for the dashboard
From: alex.chen@customer.com

Hey, love the product! One thing that would really help is a dark mode option.
No rush, just a thought.""",
    ),
]

for label, email in samples:
    result = classify(email)
    print(f"--- {label} ---")
    print(f"  Urgency:  {result.urgency}")
    print(f"  Category: {result.category}")
    print(f"  Summary:  {result.summary}")
    print(f"  Action:   {result.recommended_action}")
    print()

## 5. Provider Swap Demo

Run the same email through a different model string.

In [ ]:
email = samples[0][1]

for model in ["openai/gpt-4o-mini", "mistralai/mistral-7b-instruct"]:
    try:
        result = classify(email, model=model)
        print(f"[{model}] urgency={result.urgency} category={result.category}")
    except Exception as e:
        print(f"[{model}] error: {e}")

## 6. Starter Exercise

Extend `EmailTriage` with a `sentiment` field (`Literal["positive", "neutral", "negative"]`).  
Re-run both samples and verify the new field is populated without changing the workflow code.

In [ ]:
# Your code here
# Hint: add  sentiment: Literal["positive", "neutral", "negative"]  to the schema
# The workflow function needs no changes -- just pass the updated class as response_format

### Answer Key

In [ ]:
class EmailTriageV2(BaseModel):
    urgency: Literal["high", "medium", "low"]
    category: Literal["billing", "technical", "general", "spam"]
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        description="Overall emotional tone of the email sender."
    )
    summary: str
    recommended_action: str


def classify_v2(email_text: str, model: str = "openai/gpt-4o-mini") -> EmailTriageV2:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": "You are an email triage assistant."},
            {"role": "user", "content": email_text},
        ],
        response_format=EmailTriageV2,
    )
    return completion.choices[0].message.parsed


for label, email in samples:
    r = classify_v2(email)
    print(f"{label}: urgency={r.urgency} sentiment={r.sentiment}")